# Protein Binder Assessment & Engineering with PyRosetta
## 1. Importing Libraries/Initializing PyRosetta/Initializing Helper Functions

In [ ]:
# ============================================================
# CELL 1: Imports and PyRosetta Initialization / Define Optimization Functions
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import os, time
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol
from IPython.display import display, HTML

import pyrosetta
from pyrosetta import pose_from_pdb
from pyrosetta.rosetta.protocols.rigid import RigidBodyTransMover
from pyrosetta.rosetta.protocols.relax import FastRelax
from pyrosetta.rosetta.core.scoring import get_score_function
from pyrosetta.rosetta.core.scoring import CA_rmsd
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover

init_flags = "-mute all -ex1 -ex2 -use_input_sc -ignore_unrecognized_res"
pyrosetta.init(init_flags)

from pyrosetta.rosetta.core.select.residue_selector import (
    ResidueIndexSelector,
    NeighborhoodResidueSelector,
    NotResidueSelector
)
from pyrosetta.rosetta.core.pack.task import TaskFactory
from pyrosetta.rosetta.core.pack.task.operation import (
    InitializeFromCommandline,
    IncludeCurrent,
    NoRepackDisulfides,
    PreventRepackingRLT,
    RestrictToRepackingRLT,
    RestrictAbsentCanonicalAASRLT,
    OperateOnResidueSubset
)
from pyrosetta.rosetta.protocols.minimization_packing import PackRotamersMover

# Define the standard Rosetta score function (ref2015)
scorefxn = get_score_function()

def run_fastrelax(pose, scorefxn, repeats=3):
    """
    Runs FastRelax on the provided pose in-place.
    Compares the relaxed pose to the original and prints structural metrics.
    """
    original_pose = pose.clone()
    original_score = scorefxn(original_pose)

    relax = FastRelax(scorefxn, repeats) 
    relax.constrain_relax_to_start_coords(True)

    print(f"Running FastRelax (repeats={repeats})...")
    
    t0 = time.time()
    relax.apply(pose) 
    elapsed = time.time() - t0

    relaxed_score = scorefxn(pose)

    print(f"\nFastRelax complete in {elapsed:.0f}s")
    print(f"\n{'Metric':<30s}{'Before':>12s}{'After':>12s}{'Change':>12s}")
    print("-" * 66)
    print(f"{'Total score (REU)':<30s}{original_score:>12.1f}{relaxed_score:>12.1f}{relaxed_score - original_score:>12.1f}")

    rmsd = CA_rmsd(original_pose, pose)
    print(f"\nCα RMSD from original structure: {rmsd:.3f} Å")
    print("(< 1.0 Å means we stayed close to the experimental/starting structure)\n")
    
    return pose

def calculate_binding_energy(input_pose, scorefxn, interface="A_B"):
    """
    Estimate binding energy accurately using InterfaceAnalyzerMover.
    No more blind translations causing massive steric clashes!

    Parameters
    ----------
    input_pose : Pose - the bound complex (will not be modified)
    scorefxn   : ScoreFunction
    interface  : str - which chains to separate (e.g., "A_B" or "ABCD_E")

    Returns
    -------
    dict with bound_score, unbound_score, and binding_energy
    """
    from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover
    
    # 1. Score the bound state
    bound_pose = input_pose.clone()
    bound_score = scorefxn(bound_pose)

    # 2. Use the professional interface analyzer
    analyzer = InterfaceAnalyzerMover(interface)
    analyzer.set_pack_separated(True) # Ensures sidechains repack nicely when pulled apart
    analyzer.apply(bound_pose)
    
    # 3. Extract the accurate binding energy (dG)
    binding_energy = analyzer.get_interface_dG()

    # Calculate unbound score mathematically (Bound - dG = Unbound)
    unbound_score = bound_score - binding_energy

    return {
        "bound_score": bound_score,
        "unbound_score": unbound_score,
        "binding_energy": binding_energy
    }

def calculate_ddG(wt_pose, pose_position, new_aa, scorefxn, interface="A_B"):
    """
    Calculate ΔΔG for a point mutation using the safe InterfaceAnalyzerMover.

    Returns
    -------
    dict with wt_dG, mut_dG, ddG, and mutation label
    """
    wt_res = wt_pose.residue(pose_position).name1()
    pdb_info = wt_pose.pdb_info().pose2pdb(pose_position).strip()

    # ΔG of wild-type
    wt_result = calculate_binding_energy(wt_pose, scorefxn, interface)

    # Introduce mutation (uses your existing mutate_and_repack function)
    mut_pose = mutate_and_repack(wt_pose, pose_position, new_aa, scorefxn)

    # ΔG of mutant
    mut_result = calculate_binding_energy(mut_pose, scorefxn, interface)

    # Calculate ΔΔG (Mutant Binding Energy - Wild-Type Binding Energy)
    # A negative ddG means the mutation IMPROVED binding.
    ddG = mut_result["binding_energy"] - wt_result["binding_energy"]

    label = f"{wt_res}{pdb_info.split()[0]}{new_aa}"

    return {
        "mutation": label,
        "pdb_info": pdb_info,
        "wt_dG": wt_result["binding_energy"],
        "mut_dG": mut_result["binding_energy"],
        "ddG": ddG
    }

def mutate_and_repack(input_pose, pose_position, new_aa, scorefxn):
    """
    Introduce a point mutation and repack neighboring side chains.

    Parameters
    ----------
    input_pose    : Pose (will be cloned, not modified)
    pose_position : int - residue number in Pose numbering
    new_aa        : str - one-letter amino acid code (e.g., 'A' for Ala)
    scorefxn      : ScoreFunction

    Returns
    -------
    mutant_pose : Pose with the mutation applied
    """
    mutant_pose = input_pose.clone()

    # Select the mutation site
    mut_selector = ResidueIndexSelector(str(pose_position))

    # Select neighborhood (within 8 \u00c5) for repacking
    nbr_selector = NeighborhoodResidueSelector()
    nbr_selector.set_focus_selector(mut_selector)
    nbr_selector.set_include_focus_in_subset(True)

    not_mutsite = NotResidueSelector(mut_selector)

    # Build TaskFactory
    tf = TaskFactory()
    tf.push_back(InitializeFromCommandline())
    tf.push_back(IncludeCurrent())
    tf.push_back(NoRepackDisulfides())

    # Prevent repacking outside the neighborhood
    prevent_rlt = PreventRepackingRLT()
    tf.push_back(OperateOnResidueSubset(prevent_rlt, nbr_selector, True))

    # Restrict neighbors to repacking only (no design)
    repack_rlt = RestrictToRepackingRLT()
    tf.push_back(OperateOnResidueSubset(repack_rlt, not_mutsite))

    # Design only the target position to the desired amino acid
    restrict_aa = RestrictAbsentCanonicalAASRLT()
    restrict_aa.aas_to_keep(new_aa)
    tf.push_back(OperateOnResidueSubset(restrict_aa, mut_selector))

    # Apply packer
    packer = PackRotamersMover()
    packer.task_factory(tf)
    packer.apply(mutant_pose)

    return mutant_pose

def show_complex(pdb_path, target_chain="A", binder_chain="B"):
    with open(pdb_path) as f:
        pdb_str = f.read()

    view = py3Dmol.view(width=800, height=400)
    view.addModel(pdb_str, "pdb")

    # Target: Soft Gray (to keep focus on interface)
    view.setStyle({"chain": target_chain}, {"cartoon": {"color": "#CCCCCC"}})
    # Binder: Bright Red (to see its fold clearly)
    view.setStyle({"chain": binder_chain}, {"cartoon": {"color": "#e74c3c"}})

    view.zoomTo()
    return view.show()

## 2. Loading & Exploring the Protein Complex

In [ ]:
# ============================================================
# CELL 2: Load Target Structure
# ============================================================
# Paste your target PDB path here
pdb_file = "/home/tonypeonio/ProteinDesignChallenge/old_results/run_19/colabfold_multimer_results/nitrogenase_binder_0_TOP1_unrelaxed_rank_001_alphafold2_multimer_v3_model_1_seed_000.pdb"

if not os.path.isfile(pdb_file):
    raise FileNotFoundError(f"Could not find the PDB file at: {pdb_file}")

# Load into PyRosetta
pose = pose_from_pdb(pdb_file)
print(f"Loaded pose with {pose.total_residue()} residues ({pose.num_chains()} chains)")
print(f"Total residues: {pose.total_residue()}")

print(f"{'Chain':<8}{'Start':>6}{'End':>6}{'Length':>8}  Sequence (First 40 AA)")
print("-" * 75)
for ch in range(1, pose.num_chains() + 1):
    start = pose.chain_begin(ch)
    end = pose.chain_end(ch)
    length = end - start + 1
    chain_id = pose.pdb_info().chain(start)
    seq = pose.chain_sequence(ch)[:40]
    print(f"  {chain_id:<6}{start:>6}{end:>6}{length:>8}  {seq}...")

# Standardize Chain IDs for subsequent cells
# Usually: Target = Chain A, Binder = Chain B
TARGET_CHAIN = "A"
BINDER_CHAIN = "B"

print(f"Visualizing Complex: Target ({TARGET_CHAIN}) in Gray, Binder ({BINDER_CHAIN}) in Red")
show_complex(pdb_file, TARGET_CHAIN, BINDER_CHAIN)

---
## 3. Scoring with the Rosetta Energy Function

Rosetta evaluates structures using a weighted sum of energy terms. The **ref2015** score function is the current default for full-atom modeling. A more negative (lower) total score indicates a more favorable structure.

The score is decomposed into components:
- **Van der Waals**: `fa_atr` (attractive) + `fa_rep` (repulsive)
- **Electrostatics**: `fa_elec`
- **Solvation**: `fa_sol` (desolvation penalty)
- **Hydrogen bonds**: `hbond_sr_bb`, `hbond_lr_bb`, `hbond_bb_sc`, `hbond_sc`
- **Backbone geometry**: `rama_prepro`, `omega`, `p_aa_pp`

A bad score before FastRelax is nothing to be worried about.

In [ ]:
# ============================================================
# CELL 3: Score Energy Breakdown
# ============================================================
from pyrosetta.rosetta.core.scoring import ScoreType

total_score = scorefxn(pose)
energies = pose.energies()
weights = scorefxn.weights()

print(f"Total Rosetta Score: {total_score:.2f} REU\n")
print(f"{'Energy Term':<25s}{'Weighted Score':>15s}")
print("-" * 40)

# We only care about the heavy hitters for binder design
important_terms = [
    ScoreType.fa_atr,  # Attractive VDW
    ScoreType.fa_rep,  # Repulsive VDW (Clashes!)
    ScoreType.fa_sol,  # Solvation (Hydrophobic effect)
    ScoreType.fa_elec, # Electrostatics
    ScoreType.hbond_sr_bb, # H-bonds
    ScoreType.hbond_lr_bb,
    ScoreType.hbond_sc
]

for st in important_terms:
    w = weights[st]
    raw = energies.total_energies()[st]
    print(f"  {st.name:<23s}{w * raw:>12.3f}")

print("-" * 40)

## 4. Geometry Optimization with FastRelax

Takes around 7 min per iteration.

In [ ]:
# ============================================================
# CELL 4: Initial FastRelax (Settle the AlphaFold Clashes)
# ============================================================
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover

print("Running initial FastRelax to remove AlphaFold clashes...")
original_pose = pose.clone()

# Run the FastRelax function we defined in Cell 2
pose = run_fastrelax(pose, scorefxn, repeats=1)

# 1. INSTANTIATE the analyzer. 
analyzer = InterfaceAnalyzerMover("A_B") 

# 2. APPLY the instance to the pose
analyzer.apply(pose)
relaxed_ddg = analyzer.get_interface_dG()

print(f"\nMetrics after Initial Relax:")
print(f"Binding Energy (ddG): {relaxed_ddg:.2f} REU")


---
## 5. Interface Analysis & Binding Energy

To estimate binding energy, we:
1. **Score the bound complex** (chains together)
2. **Separate the chains** by translating them far apart
3. **Score the unbound state** (no inter-chain contacts)
4. Compute **ΔG_binding = Score_bound − Score_unbound**

A more negative ΔG indicates stronger binding.

In [ ]:
# ============================================================
# CELL 5: Calculate Binding Energy, Identify Hotspots & Destabilizing Residues
# ============================================================
import numpy as np
from pyrosetta.rosetta.protocols.rigid import RigidBodyTransMover

def get_interface_energies(input_pose, scorefxn, jump_id=1, separation=500.0):
    """
    Calculate per-residue energy changes upon binding.
    A large negative change means the residue contributes strongly to binding.
    """
    # Score bound state
    bound_pose = input_pose.clone()
    scorefxn(bound_pose)
    bound_energies = np.array([bound_pose.energies().residue_total_energy(i)
                               for i in range(1, bound_pose.total_residue() + 1)])

    # Score unbound state (Pull chains apart by 500 Angstroms)
    unbound_pose = input_pose.clone()
    trans_mover = RigidBodyTransMover(unbound_pose, jump_id)
    trans_mover.step_size(separation)
    trans_mover.apply(unbound_pose)
    scorefxn(unbound_pose)
    unbound_energies = np.array([unbound_pose.energies().residue_total_energy(i)
                                 for i in range(1, unbound_pose.total_residue() + 1)])

    # ΔE = bound - unbound (negative = stabilizing)
    return bound_energies - unbound_energies

print("Calculating per-residue binding contributions by simulating chain separation...")
delta_E = get_interface_energies(pose, scorefxn)

# --- Report top 5 stabilizing interface residues (Anchors) ---
print("\nTop 5 STABILIZING residues (Anchors - Do not mutate!):")
hotspot_idx = np.argsort(delta_E)[:5]
print(f"{'Residue':<10s}{'PDB #':>10s}{'ΔE (REU)':>12s}")
print("-" * 32)
for idx in hotspot_idx:
    res = pose.residue(idx + 1)
    pdb_num = pose.pdb_info().pose2pdb(idx + 1).strip()
    print(f"  {res.name():>5s}  {pdb_num:>8s}  {delta_E[idx]:>10.2f}")

# --- Report top 5 destabilizing interface residues (Liabilities) ---
interface_mask = np.abs(delta_E) > 0.1
destab_idx = np.argsort(delta_E[interface_mask])[::-1]
destab_positions = np.where(interface_mask)[0][destab_idx][:5]

print("\nTop 5 DESTABILIZING interface residues (Mutation Targets):")
print(f"{'Residue':<10s}{'PDB #':>10s}{'ΔE (REU)':>12s}")
print("-" * 32)
for idx in destab_positions:
    if delta_E[idx] <= 0:
        break  # Stop if we hit stabilizing residues
    res = pose.residue(idx + 1)
    pdb_num = pose.pdb_info().pose2pdb(idx + 1).strip()
    print(f"  {res.name():>5s}  {pdb_num:>8s}  {delta_E[idx]:>+10.2f}")

In [ ]:
# ============================================================
# Visualize interface hotspots in 3D
# ============================================================

import os
import py3Dmol

# Save a temporary file, read it, and delete it immediately!
temp_pdb = "temp_viz_delete_me.pdb"
pose.dump_pdb(temp_pdb)

with open(temp_pdb) as f:
    pdb_str = f.read()

os.remove(temp_pdb) #

view = py3Dmol.view(width=700, height=500)
view.addModel(pdb_str, "pdb")

# Base style: transparent cartoon
view.setStyle({"cartoon": {"color": "white", "opacity": 0.4}})

# Highlight hotspot residues (top contributors to binding)
# Color by energy contribution: blue = stabilizing, red = destabilizing
for idx in range(len(delta_E)):
    if abs(delta_E[idx]) > 1.5:  # Only show significant contributors
        pose_num = idx + 1
        chain = pose.pdb_info().chain(pose_num)
        pdb_resnum = pose.pdb_info().number(pose_num)
        if delta_E[idx] < 0:
            color = "#2980b9"  # Blue for stabilizing
        else:
            color = "#c0392b"  # Red for destabilizing
        view.addStyle(
            {"chain": chain, "resi": pdb_resnum},
            {"stick": {"color": color}, "cartoon": {"color": color, "opacity": 0.9}}
        )

view.zoomTo()
view.show()
print("Blue sticks = stabilizing interface residues | Red sticks = destabilizing")

# PackRotamersMover
Takes around 20 minutes.

### Helpful One-Letter Amino Acid Codes

```
A=Ala  C=Cys  D=Asp  E=Glu  F=Phe  G=Gly  H=His  I=Ile  K=Lys  L=Leu
M=Met  N=Asn  P=Pro  Q=Gln  R=Arg  S=Ser  T=Thr  V=Val  W=Trp  Y=Tyr
```

In [ ]:
# ============================================================
# FINAL AUTOMATED PIPELINE: Interface Design, Relax, & Export
# ============================================================
import os
import time
from pyrosetta.rosetta.core.pack.task import TaskFactory
from pyrosetta.rosetta.core.pack.task.operation import OperateOnResidueSubset, RestrictToRepackingRLT, PreventRepackingRLT
from pyrosetta.rosetta.core.select.residue_selector import ChainSelector, NeighborhoodResidueSelector, AndResidueSelector, NotResidueSelector
from pyrosetta.rosetta.protocols.minimization_packing import PackRotamersMover
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover
from pyrosetta.rosetta.core.io.pdb import dump_pdb

print("STARTING AUTOMATED BINDER OPTIMIZATION...")
t_start = time.time()

# ------------------------------------------------------------
# 1. SMART TASK FACTORY SETUP
# ------------------------------------------------------------
target_sel = ChainSelector(TARGET_CHAIN)
binder_sel = ChainSelector(BINDER_CHAIN)
interface_sel = NeighborhoodResidueSelector(target_sel, 8.0, True)

binder_interface = AndResidueSelector(binder_sel, interface_sel)
binder_core = AndResidueSelector(binder_sel, NotResidueSelector(interface_sel))

tf = TaskFactory()
tf.push_back(pyrosetta.rosetta.core.pack.task.operation.InitializeFromCommandline())
tf.push_back(pyrosetta.rosetta.core.pack.task.operation.IncludeCurrent())

tf.push_back(OperateOnResidueSubset(PreventRepackingRLT(), target_sel))
tf.push_back(OperateOnResidueSubset(RestrictToRepackingRLT(), binder_core))

# ------------------------------------------------------------
# 2. RUN THE PACKER (Automated Mutagenesis)
# ------------------------------------------------------------
print("\nRunning Packer (Simulated Annealing on Binder Interface)...")
packer_task = tf.create_task_and_apply_taskoperations(pose)
packer = PackRotamersMover(scorefxn, packer_task)
packer.apply(pose)

# ------------------------------------------------------------
# 3. HEAVY FAST-RELAX (Settle the new mutations)
# ------------------------------------------------------------
print(f"\nRunning Heavy FastRelax (5 repeats). This will take a moment...")
pose = run_fastrelax(pose, scorefxn, repeats=5)

# ------------------------------------------------------------
# 4. FINAL METRICS & VERIFICATION
# ------------------------------------------------------------
print("\nCalculating Final Interface Metrics...")
iam = InterfaceAnalyzerMover(f"{TARGET_CHAIN}_{BINDER_CHAIN}")
iam.set_scorefunction(scorefxn)
iam.set_compute_packstat(True)
iam.set_compute_interface_energy(True)
iam.apply(pose)

In [ ]:
# ============================================================
# IMMEDIATE RESCUE & EXPORT 
# ============================================================
import os
from pyrosetta.rosetta.core.io.pdb import dump_pdb

# 1. Properly extract the stats with the [1] restored!
final_ddg = iam.get_interface_dG()
final_dsasa = iam.get_all_data().dSASA[1]  

print("\nFinal Interface Metrics...")
print("-" * 45)
print(f"{'Binding Energy (ddG)':<30s}{final_ddg:>11.2f} REU")
print(f"{'Buried Surface Area':<30s}{final_dsasa:>11.0f} Å²")
print("-" * 45)

# 2. Export the files safely
OUTPUT_DIR = "../pyrosetta_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Safely extract the exact filename
base_name = os.path.splitext(os.path.basename(pdb_file))[0] 
output_pdb = os.path.join(OUTPUT_DIR, f"{base_name}_OPTIMIZED.pdb")
output_fasta = os.path.join(OUTPUT_DIR, f"{base_name}_OPTIMIZED.fasta")

dump_pdb(pose, output_pdb)

new_seq = ""
for i in range(1, pose.num_chains() + 1):
    if pose.pdb_info().chain(pose.chain_begin(i)) == BINDER_CHAIN:
        new_seq = pose.chain_sequence(i)
        break

with open(output_fasta, "w") as f:
    f.write(f">{base_name}_Optimized_Binder\n")
    f.write(new_seq + "\n")

print(f"\nRESCUE COMPLETE!")
print(f"PDB securely saved to:   {output_pdb}")

# Extract and Prepare Full FASTA

In [ ]:
# ============================================================
# EXPORT FULL COMPLEX FASTA (WILDTYPE + OPTIMIZED BINDER)
# ============================================================
import os
from pyrosetta import pose_from_pdb

local_pdb_path = "pdb/7ut8.pdb" 
print(f"Loading wildtype nitrogenase from {local_pdb_path}...")
wt_pose = pose_from_pdb(local_pdb_path)

# Set up the output directory
OUTPUT_DIR = "../pyrosetta_outputs/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

base_name = os.path.splitext(os.path.basename(pdb_file))[0] 
output_complex_fasta = os.path.join(OUTPUT_DIR, f"{base_name}_FULL_COMPLEX.fasta")

print(f"Assembling final FASTA with WT Nitrogenase and Optimized Binder...")

with open(output_complex_fasta, "w") as f:
    # 1. Write the cleaned wildtype nitrogenase chains
    for i in range(1, wt_pose.num_chains() + 1):
        chain_letter = wt_pose.pdb_info().chain(wt_pose.chain_begin(i))
        seq = wt_pose.chain_sequence(i)
        
        # Skip ligand/metal chains (all 'Z's)
        if all(aa == 'Z' for aa in seq):
            continue
            
        # Strip trailing 'Z's from protein chains
        clean_seq = seq.replace('Z', '')
        
        if len(clean_seq) > 0:
            f.write(f">Wildtype_Nitrogenase_Chain_{chain_letter}\n")
            f.write(clean_seq + "\n")
            
    # 2. Extract and write the newly optimized binder from your active `pose`
    for i in range(1, pose.num_chains() + 1):
        if pose.pdb_info().chain(pose.chain_begin(i)) == BINDER_CHAIN:
            binder_seq = pose.chain_sequence(i)
            f.write(">Optimized_Binder_Chain_New\n")
            f.write(binder_seq + "\n")
            break

print(f"\nSUCCESS!")
print(f"Full complex FASTA safely exported to: {output_fasta}")